# Patient-Specific Seizure Detection — LORO Evaluation

Fixed-hyperparameter pipeline (no SVM tuning). For each patient:
LORO over seizure records, per-fold operating-point calibration on
out-of-fold training scores, SzCORE event-level scoring.

**Set your hyperparameters once in the CONFIG cell.** AP / ROC-AUC are
reported per fold as leakage-free diagnostics only — they select nothing.

## Imports

In [60]:
pip install timescoring

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
# single-thread BLAS so the outer joblib parallelism over folds is clean
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import hashlib
from joblib import Parallel, delayed

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (LeaveOneGroupOut, GroupKFold,
                                     cross_val_predict)
from sklearn.metrics import roc_auc_score, average_precision_score

from timescoring import scoring
from timescoring.annotations import Annotation

## CONFIG

Everything you change between runs lives here. The SVM hyperparameters are
**fixed** — set them once; nothing is tuned on test data.

In [2]:
# --- patients to run ---
# Sharding: set PATIENT_SHARD env var to run a subset across separate sessions.
#   PATIENT_SHARD="chb04"            -> just the heavy patient, its own session
#   PATIENT_SHARD="chb01,chb02,..."  -> a comma-separated list
#   unset / empty                    -> all 24
import os as _os
_ALL = [f"chb{n:02d}" for n in range(1, 25)]   # chb01 .. chb24
_shard = _os.environ.get("PATIENT_SHARD", "").strip()
if _shard:
    PATIENTS = [p.strip() for p in _shard.split(",") if p.strip()]
else:
    PATIENTS = _ALL

# --- paths ---
SEIZURE_DIR     = Path("Features/Seizure")
NON_SEIZURE_DIR = Path("Features/Non_Seizure")
ANNOTATIONS_CSV = "seizure_annotations.csv"
FEATURE_KEY     = "all_feats"      # dataset name inside the .mat files
LABEL_KEY       = "all_labels"
N_FEATURES      = 1530             # used only to detect transposed arrays

# --- FIXED SVM configs (pre-specified; reported transparently, NOT selected) ---
# Each is run in full over all patients. Add/remove rows here.
# Keep "name" short and unique — it tags every output file and row.
CONFIGS = [
    # softer boundary, more regularization
    {"name": "A10", "C": 0.1, "gamma": "scale", "percentile": 10},
     {"name": "B10", "C": 0.1,  "gamma": 0.01, "percentile": 10},
     {"name": "C10", "C": 0.1,  "gamma": 0.001, "percentile": 10},
     {"name": "D10", "C": 0.1,  "gamma": 0.0001, "percentile": 10},
     {"name": "E10", "C": 1,  "gamma": "scale", "percentile": 10},
     {"name": "F10", "C": 1,  "gamma": 0.01,   "percentile": 10},
     {"name": "G10", "C": 1,  "gamma": 0.001,   "percentile": 10},
     {"name": "H10", "C": 1,  "gamma": 0.0001,   "percentile": 10},
     {"name": "I10", "C": 10,  "gamma": "scale", "percentile": 10},
     {"name": "J10", "C": 10,  "gamma": 0.01,   "percentile": 10},
     {"name": "K10", "C": 10,  "gamma": 0.001,   "percentile": 10},
     {"name": "L10", "C": 10,  "gamma": 0.0001,   "percentile": 10},
     
 ]
# --- windowing (must match feature extraction) ---
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# --- operating-point calibration (threshold + persistence k) ---
THRESH_GRID       = None                       # None -> per-fold score quantiles
K_GRID            = [1, 2, 3, 4, 6, 8]         # consecutive-window persistence
FP_PER_DAY_BUDGET = 5                           # a-priori FP/day ceiling

# Inner CV used ONLY to pick the operating point (thr + k) per outer fold,
# computed on records that EXCLUDE the held-out test record -> leakage-free.
# GroupKFold keeps every record contributing to calibration (~5 fits/fold);
# set to None for full inner leave-one-record-out (clean but ~N fits/fold).
INNER_CAL_SPLITS  = 3

# --- SzCORE event-scoring parameters (Dan et al., 2024) ---
SZCORE_PARAMS = scoring.EventScoring.Parameters(
    toleranceStart=30, toleranceEnd=60, minOverlap=0,
    maxEventDuration=5 * 60, minDurationBetweenEvents=90,
)

# --- non-seizure TRAINING subsampling (speed; TRAINING ONLY, never test) ---
# Cap the interictal (label-0) windows used for TRAINING. Each non-seizure
# record contributes a CONTIGUOUS block (proportional to its length) so its
# windows stay in real-time order -- event-level calibration on these records
# (windows_to_mask / duration_of) then stays valid. ALL seizure windows are
# kept in full. Test records are scored in full elsewhere, so FP/day
# denominators are unaffected. None -> use all interictal windows.
# Deterministic per-record seed, stable across processes/shards.
MAX_NONSEIZ_TRAIN_WINDOWS = 10000      # e.g. 20000, or None to disable
SUBSAMPLE_SEED            = 0          # base seed (combined per record)

# --- parallelism over LORO folds ---
OUTER_N_JOBS = 8

# --- outputs ---
SUMMARY_CSV = "RESULTS_SUMMARY.csv"   # one combined summary, tagged by config

SEIZ = pd.read_csv(ANNOTATIONS_CSV)


def make_pipe(cfg):
    # Memory caching removed on purpose (it wrote a growing 'cache/' folder
    # to disk and gave no speedup). cache_size below is the in-RAM libsvm
    # kernel buffer, per worker — not disk.
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif, percentile=cfg["percentile"])),
        ("svm",    SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"],
                       class_weight={0:1,1:100}, cache_size=5000)),
    ])

## Load one patient

Returns `seizure_records` and `non_records` dicts: `{record_name: (X, y)}`.

In [63]:
IMAG_TOL = 1e-6   # |imag| above this = invalid GC window, dropped


def _read_feats(h, key):
    """Read a features array, handling MATLAB-complex compound dtype.
    Returns (X_real_oriented_rows=windows, bad_window_mask)."""
    X = h[key][:]
    if X.dtype.names and "imag" in X.dtype.names:
        real, imag = X["real"], X["imag"]
    else:
        real, imag = X, np.zeros_like(X)
    if real.shape[0] == N_FEATURES:          # orient rows = windows
        real, imag = real.T, imag.T
    bad = np.abs(imag).max(axis=1) > IMAG_TOL
    return np.ascontiguousarray(real, dtype=np.float64), bad


def load_patient(patient):
    records = {}

    for f in sorted(SEIZURE_DIR.glob(f"{patient}*_seizure.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
                yv = h[LABEL_KEY][:].squeeze()
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if yv.shape[0] == X.shape[0]:
            y = yv.astype(int)
        else:
            y = yv[: X.shape[0]].astype(int)
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X, y = X[~bad], y[~bad]
        records[f.stem.replace("_seizure", "")] = (X, y)

    for f in sorted(NON_SEIZURE_DIR.glob(f"{patient}*_features.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X = X[~bad]
        y = np.zeros(X.shape[0], dtype=int)
        records[f.stem.replace("_features", "")] = (X, y)

    seizure_records = {k: v for k, v in records.items() if v[1].sum() > 0}
    non_records     = {k: v for k, v in records.items() if v[1].sum() == 0}
    return seizure_records, non_records

## Event-scoring & calibration helpers

In [64]:
def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN


def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1: i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n


def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask


_REF_MASK_CACHE = {}

def event_counts(record_scores, threshold, k):
    # ref_mask_from_record never depends on (threshold, k) -- only on the
    # record's ground-truth annotations and its sample-grid length n.
    # calibrate() calls this ~80-150x per patient/config with the SAME
    # record_scores (same n per record each time), so caching by
    # (record, n) avoids rebuilding an identical mask from the annotations
    # CSV every single combo. Pure speed fix -- ref mask is ground truth,
    # not a result, so caching it changes nothing about correctness.
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        key = (record, n)
        ref = _REF_MASK_CACHE.get(key)
        if ref is None:
            ref = ref_mask_from_record(record, n)
            _REF_MASK_CACHE[key] = ref
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN


def _arming_times(sc, thr, k):
    pos = (sc >= thr).astype(int)
    arm, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            arm.append(i * STEP + WINDOW_LEN)
    return arm


def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat


def _safe_div(a, b):
    return a / b if b else float("nan")


def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1, tp, fp, fn


def calibrate(oof_record_scores):
    # Selects threshold + k (the OPERATING POINT) on out-of-fold train scores.
    # This is not model selection — the SVM is already fixed.
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 20)))
    else:
        thr_grid = THRESH_GRID

    best = best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]


## LORO for one patient

Fixed SVM. Per fold: hold out one record, compute the calibration OOF on the
remaining records (inner grouped CV that EXCLUDES the held-out record), pick
(threshold, k) from those leakage-free scores, fit the deployment model on all
remaining records, and score the held-out record. Seizure records drive
sensitivity (Loop 1); seizure-free records get the same true leave-one-record-out
treatment (Loop 2) so their false alarms count.

`ap` / `roc_auc` are computed per fold on the held-out record as **diagnostics
only** — they are printed, never used to choose anything.

In [65]:
def _stable_seed(*parts):
    """Deterministic 64-bit seed from arbitrary parts, stable ACROSS processes
    (unlike Python's hash(), which is salted per-process by PYTHONHASHSEED)."""
    s = "|".join("" if p is None else str(p) for p in parts)
    return int.from_bytes(hashlib.blake2b(s.encode(), digest_size=8).digest(),
                          "little")


def build_train(seizure_records, non_records=None, exclude=None,
                exclude_non=None):
    """Stack TRAINING data (used for model fits and OOF calibration only —
    never for scoring a test record).

    seizure_records : dict of seizure-containing records {name:(X,y)}
    non_records     : dict of seizure-free records {name:(X,y)} or None
    exclude         : seizure record name to leave OUT of training
    exclude_non     : non-seizure record name to leave OUT of training
    Groups are the record names (used for OOF calibration CV).

    If MAX_NONSEIZ_TRAIN_WINDOWS is set, the interictal (label-0) windows placed
    into TRAINING are capped at that total. Each included non-seizure record
    contributes a CONTIGUOUS block sized proportionally to its length, with a
    deterministic per-record seed. Contiguous (not scattered) keeps the windows
    in real-time order, so windows_to_mask's i*STEP mapping and duration_of stay
    valid when these records feed the event-level calibration. ALL seizure
    windows are always kept. This only thins training; test records are scored
    in full elsewhere, so sensitivity, FP/day, and the LORO record boundaries
    are unaffected.
    """
    Xs, ys, grps = [], [], []

    # all seizure (label-1) windows always kept in full
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))

    if non_records:
        included = [(name, X, y) for name, (X, y) in non_records.items()
                    if name != exclude_non]
        total_non = sum(len(y) for _, _, y in included)
        cap = MAX_NONSEIZ_TRAIN_WINDOWS

        if cap is None or total_non <= cap:
            # no thinning needed
            for name, X, y in included:
                Xs.append(X); ys.append(y); grps.extend([name] * len(y))
        else:
            # proportional per-record cap as a CONTIGUOUS block, seeded
            # deterministically per (fold, record) and stable across processes.
            for name, X, y in included:
                n_i = len(y)
                keep_i = max(1, int(round(cap * n_i / total_non)))
                if keep_i >= n_i:
                    idx = np.arange(n_i)
                else:
                    rng = np.random.default_rng(
                        _stable_seed(SUBSAMPLE_SEED, exclude, exclude_non, name))
                    start = int(rng.integers(0, n_i - keep_i + 1))
                    idx = np.arange(start, start + keep_i)   # contiguous block
                Xs.append(X[idx]); ys.append(y[idx])
                grps.extend([name] * len(idx))

    return np.vstack(Xs), np.concatenate(ys), np.array(grps)


def oof_train_record_scores(seizure_records, non_records, cfg,
                            exclude=None, exclude_non=None):
    """Out-of-fold decision scores per record over the training pool that
    EXCLUDES the held-out record. Inner CV is grouped by record (GroupKFold,
    or leave-one-record-out when groups are few), so no record scores itself
    AND the held-out record never enters any inner training fold -> the
    resulting (thr, k) is leakage-free. Used only to calibrate the operating
    point.
    """
    Xtr, ytr, grps = build_train(seizure_records, non_records,
                                 exclude=exclude, exclude_non=exclude_non)
    n_groups = len(np.unique(grps))
    if INNER_CAL_SPLITS is None or n_groups <= INNER_CAL_SPLITS:
        cv = LeaveOneGroupOut()
    else:
        cv = GroupKFold(n_splits=INNER_CAL_SPLITS)
    oof = cross_val_predict(make_pipe(cfg), Xtr, ytr, groups=grps,
                            cv=cv, method="decision_function", n_jobs=1)
    out = []
    for name in np.unique(grps):
        m = (grps == name)
        out.append((name, oof[m], ytr[m], duration_of(ytr[m])))
    return out


def diag_scores(y, sc):
    y = np.asarray(y).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return float("nan"), float("nan")
    return roc_auc_score(y, sc), average_precision_score(y, sc)


def process_seizure_fold(seizure_records, non_records, test_record, cfg):
    """LOOP 1 fold: train on all non-seizure + all-but-one seizure, test on the
    held-out seizure record. The operating point (thr, k) is calibrated on a
    nested OOF computed WITHOUT this record -> leakage-free per fold."""
    oof = oof_train_record_scores(seizure_records, non_records,
                                  exclude=test_record, cfg=cfg)
    thr, k = calibrate(oof)
    Xtr, ytr, _ = build_train(seizure_records, non_records, exclude=test_record)
    model = make_pipe(cfg).fit(Xtr, ytr)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    roc, ap = diag_scores(yte, sc)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=roc, ap=ap)


def process_nonseizure_fold(seizure_records, non_records, test_record, cfg):
    """LOOP 2 fold (TRUE LONO): train on all seizure + all-but-one non-seizure,
    test on the held-out non-seizure record. Model refit per record (no FP
    leak); (thr, k) calibrated on a nested OOF computed WITHOUT this record."""
    oof = oof_train_record_scores(seizure_records, non_records,
                                  exclude_non=test_record, cfg=cfg)
    thr, k = calibrate(oof)
    Xtr, ytr, _ = build_train(seizure_records, non_records,
                              exclude_non=test_record)
    model = make_pipe(cfg).fit(Xtr, ytr)
    Xte, yte = non_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=float("nan"), ap=float("nan"))


## Run one patient (build tables + return summary row)

In [66]:
def run_patient(patient, cfg):
    seizure_records, non_records = load_patient(patient)
    if not seizure_records:
        print(f"[{cfg['name']}|{patient}] no seizure records — skipping")
        return None, None
    print(f"[{cfg['name']}|{patient}] seizure={len(seizure_records)} "
          f"non-seizure={len(non_records)}")

    # --- LOOP 1: LORO over seizure records (sensitivity / latency) ---
    # Each fold computes its OWN calibration OOF on records that EXCLUDE the
    # held-out test record (nested) -> the operating point is leakage-free.
    seiz_results = Parallel(n_jobs=OUTER_N_JOBS)(
        delayed(process_seizure_fold)(seizure_records, non_records, r, cfg)
        for r in seizure_records.keys()
    )

    test_outputs, meta = [], {}
    for res in seiz_results:
        test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                             res["thr"], res["k"]))
        meta[res["record"]] = dict(type="seizure", roc=res["roc"], ap=res["ap"])
        print(f"  [SEIZ] {res['record']}  thr={res['thr']:.3f} k={res['k']}  "
              f"roc={res['roc']:.3f} ap={res['ap']:.3f}  "
              f"test_pos={int(res['y'].sum())}")

    # --- LOOP 2: TRUE LONO over non-seizure records (specificity / FP) ---
    if non_records:
        non_results = Parallel(n_jobs=OUTER_N_JOBS)(
            delayed(process_nonseizure_fold)(seizure_records, non_records, r, cfg)
            for r in non_records.keys()
        )
        for res in non_results:
            test_outputs.append((res["record"], res["sc"], res["y"],
                                 res["dur"], res["thr"], res["k"]))
            meta[res["record"]] = dict(type="nonseizure", roc=float("nan"),
                                       ap=float("nan"))
        print(f"  [NON] {len(non_results)} records scored (true LONO)")

    # --- per-record pass: segment table + event totals ---
    TP = FP = FN = 0
    all_latencies, segment_rows = [], []
    for record, sc, y, dur, thr, k in test_outputs:
        is_seiz = meta[record]["type"] == "seizure"
        hyp, n = windows_to_mask(sc, dur, thr, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
        if is_seiz:
            all_latencies.extend(detection_latencies(record, sc, thr, k))

        acc, spec, sens, prec, f1, stp, sfp, sfn = segment_metrics(y, sc, 0.0)
        if not is_seiz:
            acc = spec = sens = prec = f1 = np.nan
        segment_rows.append({
            "config": cfg["name"], "C": cfg["C"], "gamma": cfg["gamma"],
            "percentile": cfg["percentile"],
            "patient": patient, "record": record,
            "n_windows": int(len(y)), "n_seizure_windows": int(np.sum(y)),
            "record_length": dur, "threshold": thr, "k": k,
            "roc_auc": meta[record]["roc"], "ap": meta[record]["ap"],
            "seg_accuracy": acc, "seg_specificity": spec,
            "seg_sensitivity": sens, "seg_precision": prec, "seg_f1": f1,
            "tp": stp, "fp": sfp, "fn": sfn,
        })
        kind = "SEIZ" if is_seiz else "NON"
        print(f"    {record} done  ({kind})", flush=True)

    # --- event-level totals ---
    sens = TP/(TP+FN) if (TP+FN) else 0.0
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
    total_len_sec = sum(t[3] for t in test_outputs)
    days = total_len_sec / 86400
    med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")

    fp_day = FP / days if days else float("nan")
    print(f"  EVENT: TP={TP} FP={FP} FN={FN}  sens={sens:.3f} "
          f"prec={prec:.3f} f1={f1:.3f}  FP/day={fp_day:.3f}  "
          f"lat={med_lat:.1f}s")

    summary_row = {
        "config": cfg["name"], "C": cfg["C"], "gamma": cfg["gamma"],
        "percentile": cfg["percentile"],
        "patient": patient,
        "total_length_hours": total_len_sec / 3600,
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "precision": prec, "f1": f1,
        "FP/day": FP / days if days else float("nan"),
        "FP/hour": FP / (days * 24) if days else float("nan"),
        "latency_median": med_lat, "n_detected": len(all_latencies),
        "mean_fold_roc_auc": float(np.nanmean(
            [m["roc"] for m in meta.values()])),
        "mean_fold_ap": float(np.nanmean([m["ap"] for m in meta.values()])),
    }
    return summary_row, segment_rows


## Run all patients

Writes one `SEGMENT_PER_RECORD_<patient>.csv` per patient and appends a row to
`RESULTS_SUMMARY.csv`. Re-runs overwrite the summary cleanly (no stale rows).

In [67]:
# all_summaries, all_segments = [], []

# for cfg in CONFIGS:
#     print(f"\n{'='*60}\nCONFIG {cfg['name']}: "
#           f"C={cfg['C']} gamma={cfg['gamma']} percentile={cfg['percentile']}"
#           f"\n{'='*60}")
#     for patient in PATIENTS:
#         try:
#             srow, segrows = run_patient(patient, cfg)
#             if srow is not None:
#                 all_summaries.append(srow)
#                 all_segments.extend(segrows)
#         except Exception as e:
#             print(f"[{cfg['name']}|{patient}] FAILED: {type(e).__name__}: {e}")
#             continue

#         # --- incremental save after EVERY patient, not just at the end ---
#         # protects against crashes / timeouts / disconnects mid-run.
#         # overwrites the same two files each time -> always reflects
#         # everything completed so far.
#         pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
#         pd.DataFrame(all_segments).to_csv("SEGMENT_PER_RECORD_ALL.csv", index=False)
#         n_done = len(all_summaries)
#         print(f"  >>> SAVED [{cfg['name']}|{patient}] "
#               f"({n_done} patient-config rows written so far) "
#               f"-> {SUMMARY_CSV}", flush=True)

# # final write (redundant with the last incremental save, kept for clarity)
# summary_df = pd.DataFrame(all_summaries)
# summary_df.to_csv(SUMMARY_CSV, index=False)
# pd.DataFrame(all_segments).to_csv("SEGMENT_PER_RECORD_ALL.csv", index=False)

# print(f"\nWrote {SUMMARY_CSV} "
#       f"({summary_df['config'].nunique()} configs x "
#       f"{summary_df['patient'].nunique()} patients)")

# # quick pooled view per config (event-level), for transparent reporting
# if len(summary_df):
#     pooled = (summary_df.groupby("config")
#               .apply(lambda d: pd.Series({
#                   "TP": d.TP.sum(), "FP": d.FP.sum(), "FN": d.FN.sum(),
#                   "sensitivity": d.TP.sum()/(d.TP.sum()+d.FN.sum())
#                                  if (d.TP.sum()+d.FN.sum()) else float("nan"),
#                   "precision": d.TP.sum()/(d.TP.sum()+d.FP.sum())
#                                if (d.TP.sum()+d.FP.sum()) else float("nan"),
#                   "FP/day": d.FP.sum()/(d.total_length_hours.sum()/24)
#                             if d.total_length_hours.sum() else float("nan"),
#               }), include_groups=False)
#               .reset_index())
#     print("\nPooled event-level by config:")
#     print(pooled.to_string(index=False))
# summary_df

# Run One Patient

## Configuration A

## Configuration D

In [68]:
# CONFIGS[3]

{'name': 'D10', 'C': 1, 'gamma': 'scale', 'percentile': 10}

In [ ]:
# cfg = CONFIGS[3]   # D10
# print(cfg)

# SUMMARY_CSV = f"RESULTS_SUMMARY_{cfg['name']}_BALANCED.csv"
# SEGMENT_CSV = f"SEGMENT_PER_RECORD_{cfg['name']}_BALANCED.csv"

# all_summaries, all_segments = [], []

# print(
#     f"\n{'='*60}\n"
#     f"CONFIG {cfg['name']}: "
#     f"C={cfg['C']} gamma={cfg['gamma']} percentile={cfg['percentile']}"
#     f"\n{'='*60}"
# )

# for patient in PATIENTS:
#     try:
#         srow, segrows = run_patient(patient, cfg)

#         if srow is not None:
#             all_summaries.append(srow)
#             all_segments.extend(segrows)

#         # incremental save after every patient
#         pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
#         pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

#         print(
#             f"  >>> SAVED [{cfg['name']}|{patient}] "
#             f"({len(all_summaries)} patient rows so far) "
#             f"-> {SUMMARY_CSV}",
#             flush=True
#         )

#     except Exception as e:
#         print(f"[{cfg['name']}|{patient}] FAILED: {type(e).__name__}: {e}")
#         continue

# # final write
# summary_df = pd.DataFrame(all_summaries)
# summary_df.to_csv(SUMMARY_CSV, index=False)
# pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

# print(f"\nWrote {SUMMARY_CSV}")

# # pooled metrics
# if len(summary_df):
#     TP = summary_df.TP.sum()
#     FP = summary_df.FP.sum()
#     FN = summary_df.FN.sum()
#     total_hours = summary_df.total_length_hours.sum()

#     pooled = pd.DataFrame([{
#         "config": cfg["name"],
#         "TP": TP,
#         "FP": FP,
#         "FN": FN,
#         "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
#         "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
#         "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
#     }])

#     print("\nPooled event-level:")
#     print(pooled.to_string(index=False))

# summary_df

{'name': 'D10', 'C': 1, 'gamma': 'scale', 'percentile': 10}

CONFIG D10: C=1 gamma=scale percentile=10
[D10|chb01] seizure=7 non-seizure=35
  [SEIZ] chb01_03  thr=-0.790 k=8  roc=0.944 ap=0.334  test_pos=16
  [SEIZ] chb01_04  thr=0.998 k=1  roc=0.993 ap=0.585  test_pos=11
  [SEIZ] chb01_15  thr=-0.792 k=8  roc=0.679 ap=0.040  test_pos=16
  [SEIZ] chb01_16  thr=-0.796 k=8  roc=0.984 ap=0.504  test_pos=21
  [SEIZ] chb01_18  thr=-0.760 k=8  roc=0.990 ap=0.905  test_pos=37
  [SEIZ] chb01_21  thr=-0.792 k=6  roc=0.991 ap=0.942  test_pos=38
  [SEIZ] chb01_26  thr=-0.858 k=6  roc=0.975 ap=0.925  test_pos=41
  [NON] 35 records scored (true LONO)
    chb01_03 done  (SEIZ)
    chb01_04 done  (SEIZ)
    chb01_15 done  (SEIZ)
    chb01_16 done  (SEIZ)
    chb01_18 done  (SEIZ)
    chb01_21 done  (SEIZ)
    chb01_26 done  (SEIZ)
    chb01_01 done  (NON)
    chb01_02 done  (NON)
    chb01_05 done  (NON)
    chb01_06 done  (NON)
    chb01_07 done  (NON)
    chb01_08 done  (NON)
    chb01_09 done  (NO

## Configuration E

In [ ]:
# CONFIGS[4]

{'name': 'E10', 'C': 1, 'gamma': 0.01, 'percentile': 10}

In [ ]:
# cfg = CONFIGS[4]   # E10
# print(cfg)

# SUMMARY_CSV = f"RESULTS_SUMMARY_{cfg['name']}_BALANCED.csv"
# SEGMENT_CSV = f"SEGMENT_PER_RECORD_{cfg['name']}_BALANCED.csv"

# all_summaries, all_segments = [], []

# print(
#     f"\n{'='*60}\n"
#     f"CONFIG {cfg['name']}: "
#     f"C={cfg['C']} gamma={cfg['gamma']} percentile={cfg['percentile']}"
#     f"\n{'='*60}"
# )

# for patient in PATIENTS:
#     try:
#         srow, segrows = run_patient(patient, cfg)

#         if srow is not None:
#             all_summaries.append(srow)
#             all_segments.extend(segrows)

#         # incremental save after every patient
#         pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
#         pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

#         print(
#             f"  >>> SAVED [{cfg['name']}|{patient}] "
#             f"({len(all_summaries)} patient rows so far) "
#             f"-> {SUMMARY_CSV}",
#             flush=True
#         )

#     except Exception as e:
#         print(f"[{cfg['name']}|{patient}] FAILED: {type(e).__name__}: {e}")
#         continue

# # final write
# summary_df = pd.DataFrame(all_summaries)
# summary_df.to_csv(SUMMARY_CSV, index=False)
# pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

# print(f"\nWrote {SUMMARY_CSV}")

# # pooled metrics
# if len(summary_df):
#     TP = summary_df.TP.sum()
#     FP = summary_df.FP.sum()
#     FN = summary_df.FN.sum()
#     total_hours = summary_df.total_length_hours.sum()

#     pooled = pd.DataFrame([{
#         "config": cfg["name"],
#         "TP": TP,
#         "FP": FP,
#         "FN": FN,
#         "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
#         "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
#         "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
#     }])

#     print("\nPooled event-level:")
#     print(pooled.to_string(index=False))

# summary_df

## Configuration F

In [ ]:
CONFIGS[5]

{'name': 'F10', 'C': 1, 'gamma': 0.1, 'percentile': 10}

In [ ]:
cfg = CONFIGS[5]   # F10
print(cfg)

SUMMARY_CSV = f"RESULTS_SUMMARY_{cfg['name']}_BALANCED.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{cfg['name']}_BALANCED.csv"

all_summaries, all_segments = [], []

print(
    f"\n{'='*60}\n"
    f"CONFIG {cfg['name']}: "
    f"C={cfg['C']} gamma={cfg['gamma']} percentile={cfg['percentile']}"
    f"\n{'='*60}"
)

for patient in PATIENTS:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        # incremental save after every patient
        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(
            f"  >>> SAVED [{cfg['name']}|{patient}] "
            f"({len(all_summaries)} patient rows so far) "
            f"-> {SUMMARY_CSV}",
            flush=True
        )

    except Exception as e:
        print(f"[{cfg['name']}|{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "config": cfg["name"],
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df

## Configuration G

In [ ]:
CONFIGS[6]

{'name': 'G10', 'C': 10, 'gamma': 'scale', 'percentile': 10}

In [ ]:
cfg = CONFIGS[6]   # G10
print(cfg)

SUMMARY_CSV = f"RESULTS_SUMMARY_{cfg['name']}_BALANCED.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{cfg['name']}_BALANCED.csv"

all_summaries, all_segments = [], []

print(
    f"\n{'='*60}\n"
    f"CONFIG {cfg['name']}: "
    f"C={cfg['C']} gamma={cfg['gamma']} percentile={cfg['percentile']}"
    f"\n{'='*60}"
)

for patient in PATIENTS:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        # incremental save after every patient
        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(
            f"  >>> SAVED [{cfg['name']}|{patient}] "
            f"({len(all_summaries)} patient rows so far) "
            f"-> {SUMMARY_CSV}",
            flush=True
        )

    except Exception as e:
        print(f"[{cfg['name']}|{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "config": cfg["name"],
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df

## Configuration H

In [ ]:
CONFIGS[7]

{'name': 'H10', 'C': 10, 'gamma': 0.01, 'percentile': 10}

In [ ]:
cfg = CONFIGS[7]   # H10
print(cfg)

SUMMARY_CSV = f"RESULTS_SUMMARY_{cfg['name']}_BALANCED.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{cfg['name']}_BALANCED.csv"

all_summaries, all_segments = [], []

print(
    f"\n{'='*60}\n"
    f"CONFIG {cfg['name']}: "
    f"C={cfg['C']} gamma={cfg['gamma']} percentile={cfg['percentile']}"
    f"\n{'='*60}"
)

for patient in PATIENTS:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        # incremental save after every patient
        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(
            f"  >>> SAVED [{cfg['name']}|{patient}] "
            f"({len(all_summaries)} patient rows so far) "
            f"-> {SUMMARY_CSV}",
            flush=True
        )

    except Exception as e:
        print(f"[{cfg['name']}|{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "config": cfg["name"],
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df

## Configuration I

In [ ]:
CONFIGS[8]

{'name': 'I10', 'C': 10, 'gamma': 0.1, 'percentile': 10}

In [ ]:
cfg = CONFIGS[8]   # H10
print(cfg)

SUMMARY_CSV = f"RESULTS_SUMMARY_{cfg['name']}_BALANCED.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{cfg['name']}_BALANCED.csv"

all_summaries, all_segments = [], []

print(
    f"\n{'='*60}\n"
    f"CONFIG {cfg['name']}: "
    f"C={cfg['C']} gamma={cfg['gamma']} percentile={cfg['percentile']}"
    f"\n{'='*60}"
)

for patient in PATIENTS:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        # incremental save after every patient
        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(
            f"  >>> SAVED [{cfg['name']}|{patient}] "
            f"({len(all_summaries)} patient rows so far) "
            f"-> {SUMMARY_CSV}",
            flush=True
        )

    except Exception as e:
        print(f"[{cfg['name']}|{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "config": cfg["name"],
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df